
## Q1. A finance analyst should see Gold data but not customer email addresses. How would you implement this?

**Important Concept: Column Masking**

**Business requirement:**

The finance team needs access to customer and sales information, but customer email addresses are confidential.

**Solution: Unity Catalog Column Masking**

### Step 1: Grant access to the Finance group

```sql
GRANT USE CATALOG
ON CATALOG workspace
TO `finance_team`;

GRANT USE SCHEMA
ON SCHEMA workspace.ecommerce_gold
TO `finance_team`;

GRANT SELECT
ON SCHEMA workspace.ecommerce_gold
TO `finance_team`;
```

### Step 2: Create a masking function

```sql
CREATE OR REPLACE FUNCTION
workspace.ecommerce_gold.mask_email(email STRING)

RETURNS STRING

RETURN CASE
    WHEN IS_ACCOUNT_GROUP_MEMBER('finance_admins')
        THEN email

    ELSE CONCAT(
        'XXXXXX',
        SUBSTRING(email, INSTR(email, '@'))
    )
END;
```

### Step 3: Apply the column mask

```sql
ALTER TABLE workspace.ecommerce_gold.customers
ALTER COLUMN Email
SET MASK workspace.ecommerce_gold.mask_email;
```

### Step 4: Test the masking

```sql
SELECT
    CustomerID,
    CustomerName,
    Email
FROM workspace.ecommerce_gold.customers;
```

**Expected output:**

| CustomerID | CustomerName | Email |
|---|---|---|
| 101 | John | XXXXXX@gmail.com |
| 102 | David | XXXXXX@yahoo.com |

### Interview Answer

I would use Unity Catalog column masking. First, I would create a SQL masking function based on group membership and apply it to the Email column. Finance analysts could access the Gold table but would see masked email addresses, while authorized administrators could see the original values.

---

## Q2. A regional manager should see only Chennai orders. How would you implement row-level security?

**Important Concept: Row Filters**

**Business requirement:**

The company operates in Chennai, Bangalore and Mumbai. Each regional manager should access only their region's orders.

**Solution: Unity Catalog Row Filters**

### Step 1: Create a row-filter function

```sql
CREATE OR REPLACE FUNCTION
workspace.ecommerce_gold.region_filter(region STRING)

RETURNS BOOLEAN

RETURN CASE
    WHEN IS_ACCOUNT_GROUP_MEMBER('ecommerce_admins')
        THEN TRUE

    WHEN IS_ACCOUNT_GROUP_MEMBER('chennai_team')
         AND region = 'Chennai'
        THEN TRUE

    WHEN IS_ACCOUNT_GROUP_MEMBER('bangalore_team')
         AND region = 'Bangalore'
        THEN TRUE

    WHEN IS_ACCOUNT_GROUP_MEMBER('mumbai_team')
         AND region = 'Mumbai'
        THEN TRUE

    ELSE FALSE
END;
```

### Step 2: Apply the row filter

```sql
ALTER TABLE workspace.ecommerce_gold.daily_sales

SET ROW FILTER
workspace.ecommerce_gold.region_filter
ON (Region);
```

### Step 3: Query the table

```sql
SELECT *
FROM workspace.ecommerce_gold.daily_sales;
```

**Expected result for a Chennai manager:**

| OrderID | Region | Amount |
|---|---|---|
| 101 | Chennai | 500 |
| 105 | Chennai | 800 |

Bangalore and Mumbai records will not be returned to the Chennai manager.

### Interview Answer

I would implement row-level security using Unity Catalog row filters. I would create a SQL function based on group membership and region, then attach that function to the table. This ensures each regional manager sees only authorized records.

---

## Q3. Your organization has MySQL data and wants to query it without copying it. What would you implement?

**Important Concept: Lakehouse Federation**

**Business requirement:**

The company stores orders in MySQL. Data analysts want to query those orders from Databricks without copying the entire database.

**Solution: Lakehouse Federation using a Connection and Foreign Catalog**

### Step 1: Create a MySQL connection

```sql
CREATE CONNECTION mysql_connection

TYPE MYSQL

OPTIONS (
    host 'mysql-server.example.com',
    port '3306',
    user 'databricks_user',
    password secret('ecommerce_scope', 'mysql_password')
);
```

### Step 2: Create a Foreign Catalog

```sql
CREATE FOREIGN CATALOG mysql_ecommerce

USING CONNECTION mysql_connection

OPTIONS (
    database 'ecommerce_db'
);
```

### Step 3: View remote schemas

```sql
SHOW SCHEMAS IN mysql_ecommerce;
```

### Step 4: View remote tables

Use the actual schema returned by the previous command.

```sql
SHOW TABLES IN mysql_ecommerce.ecommerce_db;
```

### Step 5: Query external data

```sql
SELECT *
FROM mysql_ecommerce.ecommerce_db.orders;
```

### Interview Answer

I would use Databricks Lakehouse Federation. First, I would create a MySQL connection and then register a Foreign Catalog. This allows users to query external MySQL tables through Databricks SQL without initially copying the data into Delta tables.

---

## Q4. What is the difference between a Foreign Catalog and an External Location?

**Business requirement:**

Your company has:

- MySQL customer information.
- CSV files stored in Azure Data Lake Storage.

Which Unity Catalog objects would you use?

**Answer:**

| Foreign Catalog | External Location |
|---|---|
| Connects to external databases | Controls cloud storage access |
| MySQL, PostgreSQL | S3, ADLS, GCS |
| Uses a Connection | Uses a Storage Credential |
| Exposes remote schemas and tables | Registers a cloud storage path |
| Supports federated queries | Supports governed file access |

### Example 1: Foreign Catalog

```sql
SELECT *
FROM mysql_ecommerce.ecommerce_db.customers;
```

### Example 2: External Location

```sql
CREATE EXTERNAL LOCATION ecommerce_location

URL 'abfss://landing@storageaccount.dfs.core.windows.net/ecommerce'

WITH (
    STORAGE CREDENTIAL ecommerce_storage_credential
);
```

**Note:** The Storage Credential must already exist, and the user creating the External Location must have the required privileges.

### Interview Answer

I use a Foreign Catalog when I need to access tables in external databases without initially copying their data. I use an External Location when I need to govern access to cloud storage paths using Unity Catalog storage credentials.

**Remember:**

- Foreign Catalog = External database.
- External Location = Cloud storage path.

---

## Q5. A user can see a catalog but cannot query a table. How would you troubleshoot the permissions?

**Important Concept: Access Troubleshooting**

**Business requirement:**

A data analyst can see the workspace catalog but receives a permission error when querying:

```sql
SELECT *
FROM workspace.ecommerce_gold.daily_sales;
```

**Possible reasons:**

1. Missing USE CATALOG.
2. Missing USE SCHEMA.
3. Missing SELECT.
4. Access is denied through another applicable security policy.
5. The user is not a member of the required group.

### Step 1: Check catalog permissions

```sql
SHOW GRANTS ON CATALOG workspace;
```

### Step 2: Check schema permissions

```sql
SHOW GRANTS ON SCHEMA workspace.ecommerce_gold;
```

### Step 3: Check table permissions

```sql
SHOW GRANTS ON TABLE
workspace.ecommerce_gold.daily_sales;
```

### Step 4: Grant the required privileges

```sql
GRANT USE CATALOG
ON CATALOG workspace
TO `data_analysts`;

GRANT USE SCHEMA
ON SCHEMA workspace.ecommerce_gold
TO `data_analysts`;

GRANT SELECT
ON TABLE workspace.ecommerce_gold.daily_sales
TO `data_analysts`;
```

### Step 5: Test again

```sql
SELECT *
FROM workspace.ecommerce_gold.daily_sales;
```

### Interview Answer

First, I would check the user's effective permissions at catalog, schema and table levels using SHOW GRANTS. Then I would identify the missing privilege and grant the minimum required access. I would also check group membership and applicable row filters or column masks.

---

## Q6. How do you trace an incorrect Gold KPI back to its source?

**Important Concept: Data Lineage**

**Business requirement:**

The executive dashboard displays incorrect daily revenue.

You need to identify whether the issue originated in Bronze, Silver or Gold.

**Solution: Unity Catalog Data Lineage**

### Example Architecture

```text
MySQL Orders
     |
     v
Bronze Orders
     |
     v
Silver Orders
     |
     v
Gold Daily Sales
     |
     v
Executive Dashboard
```

### Investigation Steps

1. Open Catalog Explorer.
2. Select `workspace.ecommerce_gold.daily_sales`.
3. Open the Lineage tab.
4. Identify upstream tables and transformations.
5. Trace the data back to Silver and Bronze.
6. Check whether duplicates, incorrect amounts or transformation errors caused the issue.
7. Fix the transformation and rerun the affected pipeline.

### Example SQL: Validate Silver Revenue

```sql
SELECT
    OrderDate,
    COUNT(*) AS TotalOrders,
    SUM(TotalAmount) AS TotalRevenue
FROM workspace.ecommerce_silver.orders
GROUP BY OrderDate
ORDER BY OrderDate;
```

Compare the Silver aggregation with the Gold KPI.

### Interview Answer

I would use Unity Catalog data lineage to identify upstream dependencies of the Gold table. Then I would validate the Silver and Bronze records, inspect the transformation logic and identify where the incorrect calculation originated.

**Important:** Data lineage is inspected through Catalog Explorer or supported lineage interfaces. SQL aggregation is used to validate the data, not to display lineage.

---

## Q7. How do you securely connect PostgreSQL to Databricks without hardcoding credentials?

**Business requirement:**

Your company has PostgreSQL running on an external server. You need to query its data securely from Databricks.

**Solution: Unity Catalog Connection + Secret Management + Foreign Catalog**

### Step 1: Store credentials securely

Example Python secret retrieval:

```python
username = dbutils.secrets.get(
    scope="ecommerce_scope",
    key="postgres_username"
)

password = dbutils.secrets.get(
    scope="ecommerce_scope",
    key="postgres_password"
)
```

Do not print these values or place them in notebooks or GitHub.

### Step 2: Create the connection

```sql
CREATE CONNECTION postgres_connection

TYPE POSTGRESQL

OPTIONS (
    host 'postgres-server.example.com',
    port '5432',
    user secret('ecommerce_scope', 'postgres_username'),
    password secret('ecommerce_scope', 'postgres_password')
);
```

### Step 3: Create a Foreign Catalog

```sql
CREATE FOREIGN CATALOG postgres_ecommerce

USING CONNECTION postgres_connection

OPTIONS (
    database 'ecommerce_db'
);
```

### Step 4: Query PostgreSQL

Check the remote schema first:

```sql
SHOW SCHEMAS IN postgres_ecommerce;
```

Then query using the actual schema name:

```sql
SELECT *
FROM postgres_ecommerce.public.orders;
```

### Interview Answer

I would store credentials in Databricks secret management and reference the credentials through the supported SQL secret function when creating the Unity Catalog connection. Then I would register a Foreign Catalog and query the PostgreSQL tables through Lakehouse Federation.

---

## Q8. When would you use a Foreign Catalog instead of loading data into Bronze?

**Business requirement:**

Your company has 10 million customer records in MySQL. The business wants to analyze the data from Databricks.

You have two options.

### Option A: Foreign Catalog

```sql
SELECT *
FROM mysql_ecommerce.ecommerce_db.customers;
```

**Advantages:**

- No initial full data copy.
- Quick access to external tables.
- Useful for ad-hoc analysis.
- Avoids unnecessary duplication.

**Disadvantages:**

- Depends on external database availability.
- Query performance depends on the source.
- Large analytical queries can put pressure on the source database.

### Option B: Load Data into Bronze

Example JDBC batch ingestion:

```python
df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "customers")
    .option("user", username)
    .option("password", password)
    .option("driver", "com.mysql.cj.jdbc.Driver")
    .load()
)

(
    df.write
    .format("delta")
    .mode("append")
    .saveAsTable(
        "workspace.ecommerce_bronze.customers"
    )
)
```

**Note:** This is a basic batch ingestion example, not a production-ready incremental CDC implementation. Repeated append operations can create duplicate records. A production pipeline needs a reliable change-tracking mechanism and an idempotent ingestion strategy.

**Advantages:**

- Local Delta storage.
- Historical data retention.
- Better control over transformation and optimization.
- Suitable for downstream Medallion processing.

**Disadvantages:**

- Additional storage.
- Data synchronization required.
- Ingestion pipeline maintenance.

### Comparison

| Requirement | Recommended Choice |
|---|---|
| Ad-hoc querying | Foreign Catalog |
| Avoid data duplication | Foreign Catalog |
| Historical storage | Bronze Delta |
| Large-scale transformations | Delta Tables |
| Repeated dashboard queries | Gold Delta |
| Incremental CDC pipeline | Bronze Delta + CDC/MERGE |

### Interview Answer

I would choose a Foreign Catalog for ad-hoc analysis and direct access to external data without copying it. If the business requires historical storage, incremental ingestion, large-scale transformations and optimized analytics, I would ingest the data into Bronze Delta and process it through Silver and Gold.




## 10. Final Interview Revision:


| Question | Feature / Concept |
|---|---|
| Hide customer email | Column Masking |
| Restrict Chennai orders | Row Filter |
| Query MySQL without copying | Lakehouse Federation |
| Govern ADLS/S3 storage | External Location |
| Organize external database metadata | Foreign Catalog |
| Troubleshoot access | GRANT / SHOW GRANTS |
| Trace Gold KPI | Data Lineage |
| Secure PostgreSQL credentials | Secrets + Connection |
| Historical and analytical processing | Delta Lake + Medallion Architecture |


**Interview tip:**


**For every scenario, explain four things:**

1. Business Requirement – What problem are you solving?
2. Databricks Feature – Which Unity Catalog feature would you use?
3. Implementation – How would you implement it using SQL or PySpark?
4. Reason – Why is this solution better than the alternatives?

Do not memorize SQL alone. Understand the business requirement, security implications, implementation and trade-offs.